# aiperf-lowvram — Real T4 Benchmark

**Hardware:** Tesla T4 (Turing sm_75, 15 GiB VRAM) — free Google Colab tier  
**Model:** Qwen2.5-0.5B-Instruct served by vLLM  
**Methodology:** Synthetic prompts targeting exact ISL token counts, matching AIPerf conventions  

## Methodology note

This notebook follows AIPerf's benchmarking conventions:
- Synthetic prompts generated to hit exact ISL targets (not a fixed string)
- Streaming enabled to measure TTFT and ITL separately
- Results tagged with full hardware provenance
- Seed fixed at 42 for reproducibility

**Key difference from NVIDIA's published AIPerf numbers:**  
NVIDIA benchmarks on 8x H200 GPUs with concurrency 100 and ISL/OSL 1000/500.  
This notebook benchmarks on a single T4 with concurrency 1-8 and ISL/OSL 256/128.  
Results are NOT comparable — that is the point. Hardware provenance makes this explicit.

**Before running:** Runtime → Change runtime type → T4 GPU

## Cell 1 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
print('GPU:', result.stdout.strip())
if 'T4' not in result.stdout:
    print('WARNING: Expected T4. Go to Runtime → Change runtime type → GPU → T4')
else:
    print('T4 confirmed.')

## Cell 2 — Install dependencies (~4 minutes)

In [ ]:
print('Installing vLLM 0.6.6 (last stable version for sm_75)...')
subprocess.run(['pip', 'install', '-q', 'vllm==0.6.6.post1',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121'], check=True)

print('Installing aiperf-lowvram plugin...')
subprocess.run(['pip', 'install', '-q',
    'git+https://github.com/rajmyagentit-del/aiperf-lowvram.git'], check=True)

print('Installing utilities...')
subprocess.run(['pip', 'install', '-q', 'matplotlib', 'openai', 'httpx'], check=True)
print('Done.')

## Cell 3 — Hardware detection and safety guards

In [ ]:
from aiperf_lowvram.gpu import detect_gpu
from aiperf_lowvram.guards import BenchmarkConfig, run_guards

profile = detect_gpu()
cc = profile.compute_capability

print('=' * 55)
print('  Hardware Detection (aiperf-lowvram)')
print('=' * 55)
print(f'  GPU          : {profile.name}')
print(f'  Architecture : {profile.architecture}')
print(f'  Compute cap  : sm_{cc[0]}{cc[1]}' if cc else '  Compute cap  : unknown')
print(f'  VRAM         : {profile.total_memory_gib:.2f} GiB')
print(f'  BF16 native  : {profile.supports_bfloat16}')
print(f'  FP8 native   : {profile.supports_native_fp8}')
print('=' * 55)
print()

# Benchmark config matching our sweep parameters
# ISL=256, OSL=128 — conservative for T4
# These match common AIPerf small-scale configs
ISL = 256   # input sequence length in tokens
OSL = 128   # output sequence length in tokens
MODEL_SIZE_B = 0.5

config = BenchmarkConfig(
    concurrency=8,  # max we will test
    input_sequence_length=ISL,
    output_sequence_length=OSL,
    model_size_billions=MODEL_SIZE_B,
    use_fp8=False,
    use_bf16=False,
)

guard = run_guards(profile, config)
guard.print_summary()
print(f'Safe max concurrency for {MODEL_SIZE_B}B: {profile.safe_max_concurrency(MODEL_SIZE_B)}')

## Cell 4 — Generate synthetic prompts

We use synthetic prompts targeting exact ISL token counts — the same approach AIPerf uses internally. This ensures TTFT measurements reflect the actual prefill cost of the target sequence length, not whatever a hardcoded string happens to be.

In [ ]:
import random

# Word pool producing realistic BPE token distributions
# ~1.3 tokens per word on average for Qwen tokenizer
WORD_POOL = [
    "the", "model", "inference", "latency", "throughput", "token",
    "generate", "neural", "network", "attention", "transformer",
    "benchmark", "hardware", "memory", "compute", "kernel", "batch",
    "request", "response", "stream", "cache", "prefill", "decode",
    "quantization", "precision", "floating", "point", "tensor",
    "parallel", "distributed", "serving", "deployment", "optimize",
    "performance", "measurement", "metric", "evaluation", "result",
    "system", "architecture", "configuration", "parameter", "weight",
]

def generate_prompt(target_tokens: int, seed_offset: int = 0) -> str:
    """Generate a synthetic prompt targeting approximately target_tokens.
    
    Matches AIPerf methodology: synthetic content at exact ISL targets
    rather than fixed strings.
    """
    random.seed(42 + seed_offset)
    # ~0.75 words per token for typical BPE tokenizer
    target_words = max(1, int(target_tokens * 0.75))
    words = [random.choice(WORD_POOL) for _ in range(target_words)]
    return (
        "Analyze the following technical description: "
        + " ".join(words)
        + ". Explain the key implications."
    )

# Generate prompt batch — 20 prompts per concurrency level
NUM_REQUESTS = 20
PROMPTS = [generate_prompt(ISL, seed_offset=i) for i in range(NUM_REQUESTS)]

print(f'Generated {len(PROMPTS)} synthetic prompts')
print(f'Target ISL: {ISL} tokens')
print(f'Sample prompt ({len(PROMPTS[0].split())} words):')
print(f'  "{PROMPTS[0][:120]}..."')

## Cell 5 — Start vLLM server
**Wait for `Application startup complete` before running Cell 6.**

In [ ]:
import time

MODEL    = 'Qwen/Qwen2.5-0.5B-Instruct'
PORT     = 8000
BASE_URL = f'http://localhost:{PORT}'

print(f'Starting vLLM with {MODEL}')
print('dtype=float16 — T4 has no native BF16 (sm_75 pre-Ampere)')
print('-' * 55)

server_proc = subprocess.Popen(
    [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', MODEL,
        '--dtype', 'float16',          # T4 cannot use bf16
        '--gpu-memory-utilization', '0.85',
        '--max-model-len', '2048',
        '--port', str(PORT),
        '--host', '0.0.0.0',
        '--disable-log-requests',
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

for line in server_proc.stdout:
    print(line, end='')
    if 'Application startup complete' in line or 'Uvicorn running' in line:
        print('\nServer ready.')
        break

## Cell 6 — Verify server

In [ ]:
import httpx
from openai import AsyncOpenAI

for attempt in range(20):
    try:
        resp = httpx.get(f'{BASE_URL}/health', timeout=5)
        if resp.status_code == 200:
            print(f'Server healthy (attempt {attempt+1})')
            break
    except Exception:
        pass
    time.sleep(5)

from openai import OpenAI
client_sync = OpenAI(base_url=f'{BASE_URL}/v1', api_key='not-needed')
test = client_sync.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Say hello.'}],
    max_tokens=16,
)
print(f'Test response: {test.choices[0].message.content}')
print('Server confirmed working.')

## Cell 7 — Concurrency sweep

Sweeps concurrency 1 → 2 → 4 → 8.  
Each level sends 20 requests using synthetic prompts targeting ISL=256.  
TTFT and ITL measured via streaming — same method as AIPerf.

In [ ]:
import asyncio
import statistics
from openai import AsyncOpenAI

async_client = AsyncOpenAI(base_url=f'{BASE_URL}/v1', api_key='not-needed')

async def measure_request(prompt: str):
    """Measure TTFT and ITL for a single streaming request."""
    ttft = None
    token_times = []
    t0 = time.perf_counter()

    stream = await async_client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=OSL,
        stream=True,
    )

    async for chunk in stream:
        now = time.perf_counter()
        content = chunk.choices[0].delta.content if chunk.choices else None
        if content:
            if ttft is None:
                ttft = (now - t0) * 1000
            token_times.append(now)

    itl = 0.0
    if len(token_times) > 1:
        gaps = [(token_times[i] - token_times[i-1]) * 1000
                for i in range(1, len(token_times))]
        itl = statistics.mean(gaps)

    return {
        'ttft_ms': ttft or 0.0,
        'itl_ms': itl,
        'tokens': len(token_times),
        'total_ms': (time.perf_counter() - t0) * 1000,
    }


async def sweep_concurrency(level: int):
    """Run NUM_REQUESTS total at given concurrency level."""
    all_results = []
    t_wall = time.perf_counter()

    for i in range(0, NUM_REQUESTS, level):
        batch = PROMPTS[i:i+level]
        tasks = [measure_request(p) for p in batch]
        results = await asyncio.gather(*tasks)
        all_results.extend(results)

    wall = time.perf_counter() - t_wall
    total_tokens = sum(r['tokens'] for r in all_results)
    tps = total_tokens / wall

    ttfts = [r['ttft_ms'] for r in all_results]
    itls  = [r['itl_ms']  for r in all_results if r['itl_ms'] > 0]

    def pct(lst, p):
        s = sorted(lst)
        return round(s[max(0, int(len(s)*p/100)-1)], 2)

    return {
        'concurrency':    level,
        'num_requests':   NUM_REQUESTS,
        'isl_tokens':     ISL,
        'osl_tokens':     OSL,
        'ttft_ms_avg':    round(statistics.mean(ttfts), 2),
        'ttft_ms_p50':    pct(ttfts, 50),
        'ttft_ms_p99':    pct(ttfts, 99),
        'itl_ms_avg':     round(statistics.mean(itls), 2) if itls else 0.0,
        'itl_ms_p99':     pct(itls, 99) if itls else 0.0,
        'throughput_tps': round(tps, 2),
        'wall_time_s':    round(wall, 2),
        'total_tokens':   total_tokens,
        'prompt_seed':    42,
        'dtype':          'float16',
    }


CONCURRENCY_LEVELS = [1, 2, 4, 8]
sweep_results = []

print(f'Concurrency sweep: {CONCURRENCY_LEVELS}')
print(f'Model: {MODEL}')
print(f'ISL={ISL} OSL={OSL} Requests={NUM_REQUESTS} per level')
print(f'Prompts: synthetic (seed=42), matching AIPerf ISL methodology')
print('-' * 60)

for level in CONCURRENCY_LEVELS:
    print(f'  concurrency={level}...', end=' ', flush=True)
    res = await sweep_concurrency(level)
    sweep_results.append(res)
    print(f'TTFT={res["ttft_ms_avg"]}ms  '
          f'ITL={res["itl_ms_avg"]}ms  '
          f'{res["throughput_tps"]} tok/s')

print('-' * 60)
print('Sweep complete.')

## Cell 8 — Save provenance-tagged results

In [ ]:
import datetime, json, os, platform, sys
import vllm

gpu = detect_gpu()
cc  = gpu.compute_capability

envelope = {
    'plugin': 'aiperf-lowvram',
    'plugin_version': '0.1.0',
    'timestamp_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'notebook': 'notebooks/t4_benchmark.ipynb',
    'methodology': {
        'prompt_type': 'synthetic',
        'prompt_seed': 42,
        'isl_target_tokens': ISL,
        'osl_max_tokens': OSL,
        'requests_per_level': NUM_REQUESTS,
        'measurement': 'streaming TTFT and ITL',
        'note': (
            'Synthetic prompts targeting exact ISL token counts, '
            'matching AIPerf benchmarking conventions. '
            'NOT comparable to NVIDIA published H200 numbers — '
            'different hardware class, concurrency range, and ISL/OSL.'
        ),
    },
    'environment': {
        'python_version': sys.version.split()[0],
        'platform': platform.platform(),
        'vllm_version': vllm.__version__,
    },
    'hardware': gpu.provenance_dict(),
    'benchmark_config': {
        'model': MODEL,
        'dtype': 'float16',
        'gpu_memory_utilization': 0.85,
        'max_model_len': 2048,
        'concurrency_levels_tested': CONCURRENCY_LEVELS,
    },
    'results': sweep_results,
    'key_findings': {
        'best_ttft_ms': min(r['ttft_ms_avg'] for r in sweep_results),
        'best_ttft_concurrency': min(
            sweep_results, key=lambda r: r['ttft_ms_avg'])['concurrency'],
        'peak_throughput_tps': max(r['throughput_tps'] for r in sweep_results),
        'peak_throughput_concurrency': max(
            sweep_results, key=lambda r: r['throughput_tps'])['concurrency'],
        'ttft_1x_to_8x_ratio': round(
            sweep_results[-1]['ttft_ms_avg'] / sweep_results[0]['ttft_ms_avg'], 2),
        'hardware_note': (
            f'{gpu.name} sm_{cc[0]}{cc[1]} — '
            f'no native BF16, no native FP8, float16 only'
        ),
    }
}

os.makedirs('results', exist_ok=True)
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
json_path = f'results/t4_concurrency_sweep_{ts}.json'

with open(json_path, 'w') as f:
    json.dump(envelope, f, indent=2)

print(f'Saved: {json_path}')
print()
print('Key findings:')
for k, v in envelope['key_findings'].items():
    print(f'  {k}: {v}')

## Cell 9 — Plot results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

conc  = [r['concurrency']    for r in sweep_results]
ttft  = [r['ttft_ms_avg']   for r in sweep_results]
ttftp = [r['ttft_ms_p99']   for r in sweep_results]
itl   = [r['itl_ms_avg']    for r in sweep_results]
tps   = [r['throughput_tps'] for r in sweep_results]
degrade = [t/ttft[0] for t in ttft]

fig = plt.figure(figsize=(12, 8))
fig.suptitle(
    f'aiperf-lowvram — Qwen2.5-0.5B · Tesla T4 (sm_75) · float16\n'
    f'ISL={ISL} OSL={OSL} · Synthetic prompts (seed=42) · {NUM_REQUESTS} req/level',
    fontsize=12, fontweight='bold'
)
gs = gridspec.GridSpec(2, 2, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0,0])
ax1.plot(conc, ttft, 'o-', color='#4f8ef7', lw=2, label='avg')
ax1.plot(conc, ttftp, 's--', color='#e5534b', lw=1.5, label='p99')
ax1.set(xlabel='Concurrency', ylabel='TTFT (ms)', title='Time to First Token')
ax1.legend(); ax1.grid(alpha=0.3); ax1.set_xticks(conc)

ax2 = fig.add_subplot(gs[0,1])
ax2.plot(conc, itl, 'o-', color='#3ecf8e', lw=2)
ax2.set(xlabel='Concurrency', ylabel='ITL (ms)', title='Inter-Token Latency (avg)')
ax2.grid(alpha=0.3); ax2.set_xticks(conc)

ax3 = fig.add_subplot(gs[1,0])
ax3.bar(conc, tps, color='#f5a623', width=0.6)
ax3.set(xlabel='Concurrency', ylabel='Tokens/sec', title='System Throughput')
ax3.grid(alpha=0.3, axis='y'); ax3.set_xticks(conc)

ax4 = fig.add_subplot(gs[1,1])
ax4.plot(conc, degrade, 'o-', color='#9b59b6', lw=2)
ax4.axhline(1.0, color='gray', ls='--', alpha=0.5, label='baseline')
ax4.set(xlabel='Concurrency', ylabel='TTFT / baseline', title='TTFT Degradation')
ax4.legend(); ax4.grid(alpha=0.3); ax4.set_xticks(conc)

fig.text(0.5, 0.01,
    f'Hardware: {gpu.name} · sm_{cc[0]}{cc[1]} · '
    f'{gpu.total_memory_gib:.1f} GiB · '
    f'BF16={gpu.supports_bfloat16} · FP8={gpu.supports_native_fp8} · '
    f'Results NOT comparable to NVIDIA H200 published benchmarks',
    ha='center', fontsize=8, color='gray')

png_path = f'results/t4_concurrency_sweep_{ts}.png'
plt.savefig(png_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {png_path}')

## Cell 10 — Download results

In [ ]:
from google.colab import files
import glob

for f in glob.glob('results/*'):
    print(f'Downloading: {f}')
    files.download(f)

print()
print('Next steps:')
print('1. Upload JSON and PNG to your repo under results/')
print('2. Commit: "Add real T4 benchmark results — ISL=256 OSL=128 concurrency 1-8"')
print('3. Add the PNG to your README')

## Cell 11 — Stop server

In [ ]:
server_proc.terminate()
server_proc.wait()
print('Server stopped.')
print(f'TTFT range: {min(ttft):.1f}ms — {max(ttft):.1f}ms')
print(f'Peak throughput: {max(tps):.1f} tok/s')